# Ujian Tengah Semester - Graf Sains Data
## 2.2 Analytics (Maks. skor: 55)

**Nama :** Bima Setia Sugiharto

**NIM :** 24031554040

**Kelas :** 2024A

### 1. Pilih seorang user secara acak (tentukan proses acak-nya, dan harus bisa di-reproduksi), lalu
- tuliskan skrip Python dalam merekomendasikan KNN-nya
- berikan 3 item yang diiklankan untuk k = 5 pada penentuan KNN nya

Untuk menguji kinerja sistem rekomendasi secara objektif, kita memerlukan pengguna yang memiliki riwayat aktivitas di masa lalu sekaligus aktif kembali di masa depan.

Dari seluruh pengguna yang muncul di kedua periode tersebut, kita memilih satu pengguna secara acak menggunakan fungsi `numpy.random.choice` dengan menyetel `numpy.random.seed(1143)`. Penentuan nilai *seed* ini bertujuan agar proses pengacakan bersifat deterministik dan dapat diulang (*reproducible*), sehingga pengguna yang terpilih akan selalu sama persis setiap kali kode dijalankan kembali.

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

df_train = pd.read_csv("data/events_p1_training.csv")
df_test = pd.read_csv("data/events_p2_testing.csv")

# Identifikasi user yang aktif di kedua periode
train_users = set(df_train["visitorid"].unique())
test_users = set(df_test["visitorid"].unique())
common_users = sorted(list(train_users.intersection(test_users)))

print(f"Total user di training (P1): {len(train_users):,}")
print(f"Total user di testing  (P2): {len(test_users):,}")
print(f"Total user berulang (common): {len(common_users):,} pengguna")

# Pemilihan user acak dengan seed tetap agar reproducible
np.random.seed(1143)
target_user = int(np.random.choice(common_users))

train_items = sorted(list(df_train[df_train["visitorid"] == target_user]["itemid"].unique()))
test_items = sorted(list(df_test[df_test["visitorid"] == target_user]["itemid"].unique()))

print(f"\nUser acak terpilih (seed=1143): {target_user}")
print(f"Item riwayat training P1 (input rekomendasi): {train_items}")
print(f"Item aktual testing P2 (ground truth target): {test_items}")

Total user di training (P1): 28,860
Total user di testing  (P2): 28,659
Total user berulang (common): 259 pengguna

User acak terpilih (seed=1143): 414649
Item riwayat training P1 (input rekomendasi): [np.int64(330090)]
Item aktual testing P2 (ground truth target): [np.int64(869), np.int64(118551), np.int64(303828)]


### 1(a). Skrip Python Merekomendasikan K-Nearest Neighbors (KNN)

Algoritma *User-based Collaborative Filtering* (KNN) bekerja dengan cara mencari pengguna lain (*neighbors*) yang memiliki kemiripan pola interaksi dengan pengguna target berdasarkan perhitungan *Cosine Similarity*. Setelah lima tetangga terdekat ($k = 5$) dengan nilai kemiripan tertinggi ditemukan, sistem akan mengumpulkan produk-produk yang pernah disukai oleh para tetangga tersebut.

Produk-produk yang belum pernah dilihat oleh pengguna target di masa lalu kemudian diberi bobot nilai agregasi berdasarkan skor kemiripan masing-masing tetangga. Dengan cara ini, produk yang paling diminati oleh tetangga terdekat akan memperoleh skor tertinggi dan menempati peringkat teratas rekomendasi.

In [2]:
# 1(a). Skrip Python KNN Recommender

co_users = df_train[df_train["itemid"].isin(train_items)]["visitorid"].unique()
print(f"Jumlah kandidat pengguna (co-users) dengan irisan item: {len(co_users)}")

sub_df = df_train[df_train["visitorid"].isin(co_users)]
user_item_matrix = pd.crosstab(sub_df["visitorid"], sub_df["itemid"])

user_sim = cosine_similarity(user_item_matrix.loc[[target_user]], user_item_matrix)[0]
sim_series = pd.Series(user_sim, index=user_item_matrix.index).drop(target_user)

k = 5
knn = sim_series.nlargest(k)

print("\n" + "=" * 50)
print("      TOP-5 NEAREST NEIGHBORS (k = 5)")
print("=" * 50)
for rank, (u_id, score) in enumerate(knn.items(), 1):
    print(f"{rank}. User {u_id:<8} | Skor Similarity: {score:.4f}")
print("=" * 50)

item_scores = {}
for neighbor_id, similarity in knn.items():
    n_items = df_train[df_train["visitorid"] == neighbor_id]["itemid"].unique()
    for itm in n_items:
        if itm not in train_items:
            item_scores[itm] = item_scores.get(itm, 0) + similarity

recommendations = pd.Series(item_scores).sort_values(ascending=False).head(5)

print("\n" + "=" * 50)
print("     HASIL REKOMENDASI ITEM (TOP-5 REKOMENDASI)")
print("=" * 50)
for rank, (itm, score) in enumerate(recommendations.items(), 1):
    print(f"Peringkat {rank}: Item {itm:<8} | Skor Agregasi: {score:.4f}")
print("=" * 50)

Jumlah kandidat pengguna (co-users) dengan irisan item: 6

      TOP-5 NEAREST NEIGHBORS (k = 5)
1. User 671671   | Skor Similarity: 1.0000
2. User 1282419  | Skor Similarity: 1.0000
3. User 1374313  | Skor Similarity: 0.7071
4. User 1384392  | Skor Similarity: 0.5774
5. User 1208865  | Skor Similarity: 0.4472

     HASIL REKOMENDASI ITEM (TOP-5 REKOMENDASI)
Peringkat 1: Item 234808   | Skor Agregasi: 0.7071
Peringkat 2: Item 308438   | Skor Agregasi: 0.5774
Peringkat 3: Item 432478   | Skor Agregasi: 0.5774
Peringkat 4: Item 869      | Skor Agregasi: 0.4472


### 1(b). 3 Item yang Diiklankan untuk $k = 5$

Dari daftar rekomendasi berbasis lima tetangga terdekat di atas, tiga produk teratas yang dipilih untuk dijadikan materi promosi atau diiklankan kepada pengguna adalah produk dengan ID 234808, 308438, dan 432478.

Ketiga produk ini diprioritaskan karena memiliki skor agregasi kemiripan tertinggi dari preferensi para tetangga, sehingga diprediksi memiliki peluang paling besar untuk menarik minat pengguna tersebut saat diiklankan.

In [3]:
# 1(b). Mengambil 3 item teratas untuk diiklankan
top3_advertised = list(recommendations.head(3).index)

print("=" * 50)
print(f"  3 ITEM YANG DIIKLANKAN UNTUK PENGGUNA {target_user}:")
print("=" * 50)
for idx, itm in enumerate(top3_advertised, 1):
    print(f"{idx}. Item ID: {itm:<8} (Skor Rekomendasi: {recommendations[itm]:.4f})")
print("=" * 50)


  3 ITEM YANG DIIKLANKAN UNTUK PENGGUNA 414649:
1. Item ID: 234808   (Skor Rekomendasi: 0.7071)
2. Item ID: 308438   (Skor Rekomendasi: 0.5774)
3. Item ID: 432478   (Skor Rekomendasi: 0.5774)


### 2. Berdasarkan hasil rekomendasi Anda no. 1b Analytics, hitunglah:
- Hit@5
- NDCG@5

Evaluasi performa dilakukan dengan membandingkan daftar produk rekomendasi Top 5 terhadap produk yang benar-benar dikunjungi atau dibeli pengguna pada periode testing P2 (*ground truth target*):
* **Daftar Rekomendasi Top 5**: `[234808, 308438, 432478, 869]`
* **Item Aktual Testing P2 (*Ground Truth*)**: `[869, 118551, 303828]`

#### 2(a). Hit@5
Metrik Hit@5 bernilai 1 jika minimal ada satu produk pada Top 5 rekomendasi yang cocok dengan data pengujian (*ground truth*), dan bernilai 0 jika tidak ada sama sekali:
$$\text{Hit@5} = \begin{cases} 1, & \text{jika } (\text{Top 5 Rekomendasi} \cap \text{Ground Truth}) \neq \emptyset \\ 0, & \text{lainnya} \end{cases}$$

#### 2(b). NDCG@5 (Normalized Discounted Cumulative Gain)
Metrik NDCG@5 mengukur ketepatan posisi peringkat item relevan. Semakin tinggi posisi item yang cocok, semakin sedikit penalti potongan nilainya:
$$\text{DCG@5} = \sum_{i=1}^5 \frac{rel_i}{\log_2(i + 1)}$$
$$\text{IDCG@5} = \sum_{i=1}^{|\text{Ideal}|} \frac{1}{\log_2(i + 1)}$$
$$\text{NDCG@5} = \frac{\text{DCG@5}}{\text{IDCG@5}}$$
Di mana $rel_i = 1$ jika produk pada posisi peringkat ke-$i$ relevan (*hit*), dan $0$ jika tidak.

In [4]:
rec_list = list(recommendations.index)
test_set = set(test_items)

hits = [1 if itm in test_set else 0 for itm in rec_list]

print(f"Daftar Rekomendasi Top-5: {rec_list}")
print(f"Vektor Relevansi (Hits) : {hits}")

# Perhitungan Hit@5
hit_at_5 = 1.0 if any(hits) else 0.0

# Perhitungan DCG@5
print("\nRincian Posisi Ranking:")
dcg_5 = 0.0
for idx, h in enumerate(hits, 1):
    discount = np.log2(idx + 1)
    gain = h / discount
    dcg_5 += gain
    status = "RELEVAN (HIT!)" if h == 1 else "Tidak relevan "
    print(f"- Ranking {idx}: Item {rec_list[idx-1]:<8} -> {status} | Potongan nilai: {h} / log2({idx+1}) = {gain:.4f}")

ideal_hits = sorted(hits, reverse=True)
idcg_5 = sum([h / np.log2(idx + 1) for idx, h in enumerate(ideal_hits, 1)])

# Perhitungan NDCG@5
ndcg_5 = (dcg_5 / idcg_5) if idcg_5 > 0 else 0.0

print(f"\nNilai DCG@5  : {dcg_5:.4f}")
print(f"Nilai IDCG@5 : {idcg_5:.4f}")

print("=" * 50)
print("       HASIL EVALUASI METRIK REKOMENDASI")
print("=" * 50)
print(f"2(a). Nilai Hit@5  : {hit_at_5} (Sukses mendapatkan Hit)")
print(f"2(b). Nilai NDCG@5 : {ndcg_5:.4f} (Hit berada di posisi ranking 4)")
print("=" * 50)


Daftar Rekomendasi Top-5: [234808, 308438, 432478, 869]
Vektor Relevansi (Hits) : [0, 0, 0, 1]

Rincian Posisi Ranking:
- Ranking 1: Item 234808   -> Tidak relevan  | Potongan nilai: 0 / log2(2) = 0.0000
- Ranking 2: Item 308438   -> Tidak relevan  | Potongan nilai: 0 / log2(3) = 0.0000
- Ranking 3: Item 432478   -> Tidak relevan  | Potongan nilai: 0 / log2(4) = 0.0000
- Ranking 4: Item 869      -> RELEVAN (HIT!) | Potongan nilai: 1 / log2(5) = 0.4307

Nilai DCG@5  : 0.4307
Nilai IDCG@5 : 1.0000
       HASIL EVALUASI METRIK REKOMENDASI
2(a). Nilai Hit@5  : 1.0 (Sukses mendapatkan Hit)
2(b). Nilai NDCG@5 : 0.4307 (Hit berada di posisi ranking 4)


#### Hasil Evaluasi

**1. Penjelasan Nilai Hit@5 (Skor: 1.0):**  
Nilai Hit@5 bernilai 1.0 menunjukkan bahwa sistem rekomendasi sukses memprediksi produk yang relevan. Keberhasilan ini dibuktikan oleh produk dengan ID 869 yang muncul di dalam daftar Top 5 rekomendasi (pada posisi ke-4) dan terbukti benar-benar dikunjungi oleh pengguna pada periode pengujian P2. Hal ini memastikan bahwa algoritma KNN yang dibangun berhasil menangkap kecenderungan minat pengguna.

**2. Penjelasan Nilai NDCG@5 (Skor: 0.4307):**  
Nilai NDCG@5 sebesar 0.4307 (atau 43.07%) mengukur kualitas urutan peringkat dari rekomendasi yang diberikan. Karena produk yang relevan (ID 869) berada di posisi ke-4 dan bukan di posisi pertama, nilai keuntungannya mengalami penalti potongan nilai logaritmik ($1 / \log_2(5) \approx 0.4307$) terhadap kondisi ideal ($IDCG = 1.0$). Skor ini menunjukkan bahwa rekomendasi sudah tepat sasaran dan akurat di dalam rentang Top 5 meskipun produk relevan berada di peringkat ke-4.

### Rangkuman
1. **Pemilihan User**: Terpilih **User `414649`** secara deterministik dan *reproducible* menggunakan `numpy.random.seed(1143)`.
2. **KNN Recommendation ($k = 5$)**: Ditemukan 5 tetangga terdekat dengan tingkat kemiripan Cosine hingga $1.0000$.
3. **3 Item yang Diiklankan (Top 3)**:
   * **Item `234808`** (Skor Rekomendasi: 0.7071)
   * **Item `308438`** (Skor Rekomendasi: 0.5774)
   * **Item `432478`** (Skor Rekomendasi: 0.5774)
4. **Evaluasi Metrik**:
   * **`Hit@5 = 1.0`**: Rekomendasi sukses memprediksi item yang benar-benar dikunjungi pengguna pada periode testing (Item `869` di peringkat 4).
   * **`NDCG@5 = 0.4307`**: Nilai "pengurangan nilai" di peringkat ke-4 ($1 / \log_2(5)$) dibagi dengan ideal ranking 1 ($1.0$).
